In [10]:
import urllib, requests, socket, re, lxml, io, bs4, sqlite3, sqlalchemy
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
# импорт библиотек

In [11]:
url = "https://4fermer.com/skot-ptic/pticevodstvo/kury/skolko-yaic-neset-kurica-v-den.html" #ссылка на сайт с табличкой
info = requests.get(url).text
soup = BeautifulSoup(info, 'lxml')

table = soup.find_all("div", class_="post__table")

tr_list = table[0].text.split("\n")
tr_list.remove('')
tr_numpy = np.array(tr_list)
table = tr_numpy.reshape(6, 3)

df = pd.DataFrame(table)
df.head()

,0,1,2
0,"Порода, вид",Продуктивность (в год),Масса одного яйца(в граммах)
1,Леггорн,До трехсот яиц,55-65
2,Ломан Браун,До трехсот двадцати,60-65
3,Хайсекс,До трехсот десяти,60-65
4,Русская белая,Около двухсот,55-60


# Построение бизнес-модели

In [12]:
farm_id_counter = 1 # счетчик для id ферм
chicken_id_counter = 1 # счетчик для id куриц
class Chicken:
    def __init__(self, id, breed, egg_prod_rate, health_status, age, type):

        self.id = id # Айди курицы, специальный номер курицы
        self.breed = breed # Порода курицы
        self.egg_prod_rate = egg_prod_rate # Среднее количество яиц в день
        self.health_status = health_status # Состояние здоровья (good, average, poor)
        self.age = age  # Возраст курицы (в годах)
        self.type = type # Тип (перепелка, обычная курица, Страус, шоколадная курица) [задел на будущее]

        global chicken_id_counter
        chicken_id_counter += 1  # обновление id

    def lay_eggs(self): # Учет состояния здоровья: если состояние здоровья плохое, то продуктивность куры снижается, если хорошее, то повышается
        if self.health_status == "poor":
            return max(0, self.egg_prod_rate - 0.5)
        elif self.health_status == "good":
            return max(0, self.egg_prod_rate + 0.5)
        else:
            return self.egg_prod_rate

    def info(self): # Общая информация о курице
        return (f"Курица {self.id}: порода {self.breed}, "
                f"{self.egg_prod_rate} яиц/день, здоровье: {self.health_status}, возраст: {self.age}")

In [13]:
# Класс для фермы
class Farm():
    def __init__(self, id, rental_rate, location, level):

        self.id = id # Номер фермы фермы
        self.rental_rate = rental_rate # Стоимость аренды фермы (например, руб/месяц)
        self.location = location # ЛОкация фермы (координаты вида [долгота, широта])
        self.chickens = []  # Список куриц на ферме
        self.level = level # уровень подписки (xs - перепелки, m - курицы обычные, XL - страусисные фермы, Kids - игрушечная ферма с шоколадными яйцами)

        global farm_id_counter # обновление id
        farm_id_counter += 1

    def add_chicken(self, chicken): # Добавляет курицу на ферму
        self.chickens.append(chicken)

    def remove_chicken(self, chicken_id): # Удаляет курицу с фермы по её идентификатору
        for chick in self.chickens:
            if chick.id == chicken_id:
                self.chickens.remove(chick)

    def total_daily_egg_production(self):
        # Возвращает общее дневное производство яиц на ферме
        return sum(chicken.lay_eggs() for chicken in self.chickens)

    def calculate_profit(self):
        profit = self.rental_rate
        return profit

    def get_info(self): # Содержит информацию о ферме и о всех курицах на ней
        print("Номер фермы: ", self.id)
        print("Локация: ", self.location)
        print("Стоимость аренды: ", self.rental_rate)
        print("Id кур на ферме: ")
        ids = []
        for chicken in self.chickens:
            ids.append(chicken.id)
        print(ids)

In [14]:
# Симуляция бизнеса
farms_list = [] # список, где хранятся все наши фермы

# Создаем ферму и добавляем куриц
farm1 = Farm(id=1, rental_rate=500, location=[1, 2], level="m")
farms_list.append(farm1)

chicken_list_farm1 = []
for i in range(10):
    eggctivity = np.random.normal(loc=1, scale=np.sqrt(0.1)) #оценка яйкеноскости — нормальное распределение с m=1 и var=0.1
    chicken = Chicken(id=chicken_id_counter, breed="Леггорн", egg_prod_rate=eggctivity, health_status="good", age=2, type="chiken")
    farm1.add_chicken(chicken)
    print(chicken.info())


farm1.get_info()
print("\nОбщее дневное производство яиц:", farm1.total_daily_egg_production())
egg_price = 10
our_profit = 0
for farm in farms_list:
    print(f"Потенциальная дневная прибыль владельца фермы № {farm.id}:", egg_price * farm.total_daily_egg_production())
    our_profit += farm.rental_rate
print("Наша потенциальная дневная прибыль:", our_profit)

Курица 1: порода Леггорн, 0.8708817293711332 яиц/день, здоровье: good, возраст: 2
Курица 2: порода Леггорн, 1.354376237771606 яиц/день, здоровье: good, возраст: 2
Курица 3: порода Леггорн, 1.1464189914810796 яиц/день, здоровье: good, возраст: 2
Курица 4: порода Леггорн, 0.8950403155616299 яиц/день, здоровье: good, возраст: 2
Курица 5: порода Леггорн, 1.0603006302948512 яиц/день, здоровье: good, возраст: 2
Курица 6: порода Леггорн, 0.8447368344700729 яиц/день, здоровье: good, возраст: 2
Курица 7: порода Леггорн, 0.5496967554105132 яиц/день, здоровье: good, возраст: 2
Курица 8: порода Леггорн, 1.0622540420938185 яиц/день, здоровье: good, возраст: 2
Курица 9: порода Леггорн, 0.8444770005237469 яиц/день, здоровье: good, возраст: 2
Курица 10: порода Леггорн, 1.2291583092399565 яиц/день, здоровье: good, возраст: 2
Номер фермы:  1
Локация:  [1, 2]
Стоимость аренды:  500
Id кур на ферме: 
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

Общее дневное производство яиц: 14.857340846218406
Потенциальная дневная 